In [ ]:
import os
from langchain_classic.chains import RetrievalQA
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [ ]:
os.environ["GROQ_API_KEY"] = ""

In [55]:
loader = PyPDFLoader("data/Aravind_Unni_MS_resume_DS.pdf")
documents = loader.load()

In [56]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
texts = text_splitter.split_documents(documents)
texts

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-12T21:54:35+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-12T21:54:35+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': 'Aravind Unni M S - Resume', 'trapped': '/False', 'source': 'data/Aravind_Unni_MS_resume_DS.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='ARAVIND UNNI M S\nKollam, Kerala, India Phone: +91-8925109657\nEmail: aravindunni89251@gmail.com LinkedIn: linkedin.com/in/aravind-unni-5987871ab\nPROFESSIONAL SUMMARY\nM.Tech Data Science candidate with 1.5 years of experience building production-grade AI systems, scalable Python\nbackend services, and automated MLOps pipelines. Expertise spans PyTorch and TensorFlow, NLP (LLMs, RAG),AI\nagent and Agentic workflows. Proven track record of deploying robust AI solutions on AWS via Docker and FastAPI\nto

In [57]:
embeddings = HuggingFaceEmbeddings()
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [58]:
db = Chroma.from_documents(texts, embeddings)

In [59]:
retriever = db.as_retriever(
    search_type="similarity", search_kwargs={"k": 2}
)

In [60]:
qa = RetrievalQA.from_chain_type(
    llm=ChatGroq(model_name="llama-3.3-70b-versatile",temperature=0.2),
    chain_type="map_reduce",
    retriever=retriever,
    return_source_documents=True,
    verbose=True,
)

In [62]:
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatGroq(model_name="llama-3.1-8b-instant",temperature=0.2),
    retriever=retriever,
    return_source_documents=True,  
    chain_type="stuff",            
)

In [63]:
query = "how many years of experience"

result = qa.invoke({"query": query})

print("Answer:", result['result'])
print("\nSource Documents:")
for doc in result['source_documents']:
    print(f"- {doc.page_content}")



> Entering new RetrievalQA chain...

> Finished chain.
Answer: 1.5 years of experience.

Source Documents:
- ARAVIND UNNI M S
Kollam, Kerala, India Phone: +91-8925109657
Email: aravindunni89251@gmail.com LinkedIn: linkedin.com/in/aravind-unni-5987871ab
PROFESSIONAL SUMMARY
M.Tech Data Science candidate with 1.5 years of experience building production-grade AI systems, scalable Python
backend services, and automated MLOps pipelines. Expertise spans PyTorch and TensorFlow, NLP (LLMs, RAG),AI
agent and Agentic workflows. Proven track record of deploying robust AI solutions on AWS via Docker and FastAPI
to reduce overhead and improve reliability. Seeking a Data Science or AI Engineering role to drive measurable business
impact.
WORK EXPERIENCE
AI Research Intern–CorrelationLabOctober 2025 – April 2026
Kollam, India
•Developed a predictive maintenance system using transfer learning on Short-Time Fourier Transform spectrograms,
classifying 10 bearing fault states with 95 percent accuracy.
